# Teacher's Notebook — Data Analytics and Visualisation in Data-Centric Engineering

## How to use this notebook

Run this notebook top to bottom before the session to verify it executes cleanly and the charts look right.
During the session use it as your live-coding script: each **[TEACHER]** markdown cell tells you what to say next, each code cell is what you type (or copy in) at the front of the room.

Participants work in `participant_analytics.ipynb` with `snippets.ipynb` open alongside.
Your role is to build the same analysis live, explain each decision as you go, and give them a moment to attempt each step before you reveal yours.

**Timing guide**

| Section | Topic | Target time |
|---|---|---|
| 1 | Data ingestion and cleaning | 15 min |
| 2 | Interactive scatter and relationship plots | 15 min |
| 3 | Vibration normalised by RPM | 15 min |
| 4 | Operational decision dashboard | 15 min |
| — | Ethics discussion and wrap-up | 5 min |

**What participants leave with:** a repeatable pipeline — raw CSV → cleaned two-stream data → four relationship charts → normalised vibration trend → a two-panel evidence dashboard that names which turbine to prioritise and projects when it will cross the warning threshold.

## What is in the data — read this before the session

The dataset is 90 days of 10-minute wind turbine telemetry for three turbines at a shared site.
All three turbines experience the same regional weather, so any divergence between them is an asset signal, not a weather effect.

### The three turbine stories

**WT-01** is the healthy baseline. Use it as the reference throughout. Its vibration, temperature, and power curves should stay stable. The messy dataset contains a deliberate 24-hour vibration flatline on WT-01 (around day 63) to mimic a sensor dropout — participants should spot it as a data quality issue, not a machine fault.

**WT-02** starts degrading on day 30. Bearing temperature rises, vibration increases, power output drops. On day 45 the bearing seizes: RPM and power drop to zero, bearing temperature spikes to ~165°C above ambient, vibration stays elevated. After the seizure the turbine remains offline for the rest of the dataset. WT-02 is the "already failed" turbine — the budget question is whether to fix it or watch WT-03.

**WT-03** looks healthy until day 60, then begins a slow drift: vibration creeps upward relative to RPM, bearing temperature rises slightly. This is the turbine that students should prioritise. The drift is subtle enough that raw vibration alone might not flag it — which is why the normalised ratio in Section 3 is the key diagnostic.

### Injected anomalies in the messy dataset

- **WT-01 vibration flatline**: 144 consecutive rows locked to a single value (sensor dropout pattern)
- **RPM and Power nulls**: ~3 % of rows for each column, spread across all turbines (failed sensors or data path issues)
- **Bearing temperature spikes**: three rows set to 999°C (likely sensor glitch — should be removed)
- **Power spikes**: three rows set to −500 kW (impossible value — should be removed)

### What `Wind_Speed_mps` means

`Wind_Speed_mps` is the effective wind speed at each turbine after adding gusts and local wake turbulence to the regional macro wind. It is distinct from `Macro_Wind_Speed_mps` (the shared regional reading) and is the variable that directly drives RPM and power.

## Answers — what participants should conclude at each stage

Keep these in mind as you facilitate. The point is not to hand them the answers but to guide them there with the right prompts.

| Stage | What they should find |
|---|---|
| Section 1 | RPM and Power have ~3 % nulls. Bearing temperature and vibration have zero nulls but contain impossible spikes. The flatline on WT-01 is visible as a repeating constant in the vibration column. |
| Section 2 | WT-02 separates from the healthy cluster in the temp-vs-vibration scatter. The power curve for WT-02 is much flatter than WT-01 and WT-03. WT-03 is hard to see in a single scatter — the time series is the better entry point. |
| Section 3 | WT-03's vibration-per-RPM is flat for 60 days, then rises. The rolling mean makes the trend visible. The same ratio for WT-01 stays flat throughout, ruling out a weather explanation. |
| Section 4 | The trend slope for WT-03 points to a threshold crossing within the dataset window. WT-02 is already offline. The recommendation is to prioritise WT-03 — it is the only one where proactive intervention is still possible. |

---
## [TEACHER] Opening — before you write any code

*Say something like:*

> "We have three months of telemetry from a wind farm — temperature, vibration, RPM, power, gusts, turbulence, blade angle. The question at the end of the session is: one maintenance slot, one turbine, limited budget — which one do you choose, and how does the data justify that?"

> "Before we touch any of that, we are going to load it, look at what is actually there, and make some deliberate choices about what to keep and what to remove. Those choices are the first place where analysis can go wrong."

Then open a new notebook and start from scratch. Type the imports by hand — it signals to participants that there is nothing magic here, and they can follow exactly.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

---
## Section 1 — Data Ingestion and Cleaning

### [TEACHER] What to say before loading

> "We have a file called `messy_turbine_telemetry.csv`. Before I load it I want you to think: what do you expect to see in a real sensor dataset from industrial equipment? What usually goes wrong?"

Pause for 30 seconds. Common answers: missing values, sensor dropouts, timestamp gaps, unit mismatches, stale values.

> "All of those are in here. Let's go and find them."

Load the raw data without modifying anything — this is deliberate. The first cell is diagnostic only.

In [ ]:
# Find the CSV wherever this notebook is running from
csv_candidates = [
    Path('messy_turbine_telemetry.csv'),
    Path('..') / 'messy_turbine_telemetry.csv',
    Path.cwd() / 'messy_turbine_telemetry.csv',
]
for candidate in csv_candidates:
    if candidate.exists():
        csv_path = candidate
        break
else:
    raise FileNotFoundError('Run hidden/generate_mock_wind_turbine_telemetry.py first.')

df = pd.read_csv(csv_path, parse_dates=['Timestamp'])
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
print(f'Turbines: {sorted(df["Turbine_ID"].unique())}')
print(f'Date range: {df["Timestamp"].min()} to {df["Timestamp"].max()}')

### [TEACHER] After loading — what to say

> "38,880 rows. 12,960 rows per turbine — that is 90 days at 10-minute intervals. 144 readings per day, 12,960 per turbine. That is more data than you would typically look at by hand, which is why we need the charts to do the heavy lifting."

Now show them `.info()` and `.isna().sum()` so they can see the column types and missing values together.

In [ ]:
df.info()
print()
print('Missing values per column:')
print(df.isna().sum().sort_values(ascending=False).head(15))

### [TEACHER] Reading the `.info()` output — what to say

> "Two columns stand out: RPM and Power_kW. Everything else is fully populated — 38,880 non-null. Those two have about 1,100 missing each. That is roughly 3 % of records."

> "Now ask yourself: what does a missing RPM value mean? The turbine's RPM sensor failed? The data pipeline dropped the packet? Or... the turbine stopped rotating and the control system didn't write a reading?"

> "We do not know. And that ambiguity is the point. A missing value is not noise — it might be the most important row in the dataset. If we drop it, we lose that information. If we fill it (ffill, mean, whatever), we are making up data."

> "What we are going to do is split the data into two streams: one for forensic review (the rows with issues), one for statistical analysis (the clean rows). We never lose the first stream."

**The key teaching moment here:** the choice between dropping, filling, or separating is a governance decision — it affects what a downstream model or chart can see. Make sure participants write that choice down, not just the code.

In [ ]:
# Tag rows where key sensor data is missing
df['Failure_State'] = np.where(
    df[['Bearing_Temperature_C', 'Vibration_mm_s']].isna().any(axis=1).values,
    'Sensor dropout / failure',
    'Normal',
)

# Two streams: forensic and statistical
failure_records = df[df['Failure_State'] == 'Sensor dropout / failure'].copy()
analysis_df = df.dropna(subset=['Bearing_Temperature_C', 'Vibration_mm_s']).copy()
analysis_df = analysis_df.sort_values(['Turbine_ID', 'Timestamp'])

print(f'Dropout rows retained for forensic review: {len(failure_records)}')
print(f'Rows in analysis stream: {len(analysis_df)}')

### [TEACHER] After separating the streams

> "Zero rows in the forensic stream — that means bearing temperature and vibration have no actual NaN values. But we know from `.isna().sum()` that RPM and Power are missing in about 1,100 rows each. So the dropouts are in the operational sensors, not the health sensors."

> "That is actually useful. The rows with missing RPM or Power are still in `analysis_df`. We just will not divide by RPM in those rows, and we will not average Power across them. We will handle each column separately."

Now show them the IQR filter to remove the impossible spikes — temperature at 999°C, power at −500 kW.

In [ ]:
# Remove physically impossible spikes using a 3×IQR rule
# 3× rather than 1.5× because the real degradation signal (high bearing temps in WT-02)
# falls within 1.5×IQR — we only want to remove the truly impossible injected values
valid_mask = pd.Series(True, index=analysis_df.index)

for col in ['Bearing_Temperature_C', 'Vibration_mm_s']:
    Q1 = analysis_df[col].quantile(0.25)
    Q3 = analysis_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3 * IQR
    upper = Q3 + 3 * IQR
    valid_mask &= (analysis_df[col] >= lower) & (analysis_df[col] <= upper)

print(f'Rows removed by IQR filter: {(~valid_mask).sum()}')
analysis_df = analysis_df[valid_mask]
print(f'Analysis rows remaining: {len(analysis_df)}')

### [TEACHER] Why 3×IQR not 1.5×IQR

> "Standard outlier removal uses 1.5× the interquartile range. Why am I using 3×? Because the degrading turbine — WT-02 — has bearing temperatures well above the healthy range. If I use 1.5× I remove the real degradation signal. I only want to remove the impossible values — 999°C, −500 kW — that are clearly sensor glitches."

> "This is a calibration decision. There is no universal right answer. The choice should be documented and justified — not left as a magic number in the code."

**Prompt participants:** *How would you defend 3× in a real engineering report? What if someone asks why not 2×?*

Expected answer: show that the WT-02 peak bearing temperature (~80°C during degradation) falls inside 3× but outside 1.5×. You can demonstrate this quickly by checking `analysis_df['Bearing_Temperature_C'].describe()`.

In [ ]:
# Quick sanity check: are the degradation signals still in the data?
print('Bearing temperature by turbine after cleaning:')
print(analysis_df.groupby('Turbine_ID')['Bearing_Temperature_C'].describe().round(2))

### [TEACHER] What to read in the describe output

> "WT-01 max bearing temperature: around 22°C. WT-02 max: around 80°C. WT-03 max: around 26°C. Those numbers already tell a story. If this were a real dashboard, a manager could see WT-02 has a problem without any chart at all."

> "But max alone is misleading — it could be one spike on one windy night. We need to look at how those values move over time. That is Section 2."

---
## Section 2 — Interactive Anomaly Hunting via Scatter Plots

### [TEACHER] What to say entering Section 2

> "Scatter plots are the fastest way to see if two things move together. We are going to put vibration on one axis and bearing temperature on the other, colour the points by turbine, and use point size to show RPM. Four variables in one chart."

> "The key question is: does one turbine sit in a different place in this space? If the three turbines are all healthy, they should cluster together. If one is degrading, it should separate out."

> "Give participants 30 seconds to try building this before you show your version. Tell them the column names: `Bearing_Temperature_C`, `Vibration_mm_s`, `Turbine_ID`, `RPM`."

In [ ]:
# Scatter: bearing temperature vs vibration
# Points coloured by turbine, sized by RPM
# Hover includes weather context so participants can test if wind explains the spread
scatter_df = analysis_df.dropna(subset=['Bearing_Temperature_C', 'Vibration_mm_s', 'RPM']).copy()

fig = px.scatter(
    scatter_df,
    x='Bearing_Temperature_C',
    y='Vibration_mm_s',
    color='Turbine_ID',
    size='RPM',
    size_max=6,
    hover_data=[
        'Timestamp', 'RPM', 'Wind_Speed_mps',
        'Wind_Gust_mps', 'Turbulence_Intensity', 'Operational_State',
    ],
    title='Vibration vs Bearing Temperature by Turbine',
)
fig.update_layout(template='plotly_white')
fig

### [TEACHER] What to say when the scatter appears

> "Look at the right-hand side of the chart — the high temperature zone. Most of those points belong to one turbine. Zoom in there."

> "Now hover over one of those hot points. What is the Operational_State? What is the RPM? If the RPM is close to zero and the temperature is still high, that is the failure pattern — the machine stopped rotating but the bearing is still very warm."

> "The large-RPM points (biggest circles) should be spread across a range of temperatures for a healthy turbine. If you see high RPM but also unusually high temperature, that is an early warning sign rather than an obvious failure."

**Prompt:** *Which turbine has the widest spread along the temperature axis? Can weather explain that spread?* (Hover to check wind speed on the outlying points — they should not be notably windier than the rest.)

In [ ]:
# Time series: bearing temperature — keep gaps visible where data drops out.
# Use the raw df with only the impossible injected spikes removed (999°C is sensor glitch;
# the real 80°C degradation peak and 181°C seizure spike ARE the story and must stay visible).
ts_df = df[df['Bearing_Temperature_C'] < 500].copy()

fig = px.line(
    ts_df,
    x='Timestamp',
    y='Bearing_Temperature_C',
    color='Turbine_ID',
    title='Bearing Temperature Over Time',
    hover_data=['Operational_State', 'Maintenance_Flag', 'Wind_Speed_mps'],
)
fig.update_traces(connectgaps=False)
fig.update_layout(template='plotly_white', xaxis_title='Timestamp', yaxis_title='Bearing Temperature (°C)')
fig

### [TEACHER] Reading the bearing temperature time series

> "Three lines, one per turbine. In the first month they should be close together — all three machines, same weather, similar operating temperatures. Around day 30, one line starts to separate. By day 45 it spikes dramatically and then goes somewhere unexpected. Hover on that spike — what does Maintenance_Flag say?"

> "For WT-03, the line looks flat and healthy until about day 60, then it starts to rise. The rise is slow — maybe only a few degrees — which is why you would not notice it without the trend comparison in Section 3."

**The connectgaps=False argument:** *'Without this line, Plotly connects across data gaps with a straight line. That looks like a small dip when it is actually a sensor dropout lasting hours. The gap is evidence — we want to see it, not paper over it.'*

**Prompt participants:** *Use zoom to find the exact day WT-02's bearing temperature spikes. Now use hover to find the first day WT-03's temperature starts rising above WT-01. Write both dates down — you will use them in Section 4.*

### [TEACHER] Before the 4-panel relationship chart

> "So far we have looked at one or two variables at a time. Now we are going to check four relationships that should hold for a healthy turbine."

> "A healthy turbine should obey all four of these at once:"
> 1. Power output rises with wind speed (non-linearly — it is a cubic relationship in the physics, but limited by rated output)
> 2. Higher turbulence means more structural vibration
> 3. Gusts drive short-term RPM increases
> 4. Blade pitch tracks power output — as the turbine works harder, the blades angle differently

> "If a turbine obeys some of these but not others, that is where you start to diagnose the problem."

Build the four subplots next. This cell is longer — type it or show it then let participants study the output.

In [ ]:
# Four-panel relationship dashboard
relationship_df = analysis_df.dropna(
    subset=['Wind_Speed_mps', 'Power_kW', 'Vibration_mm_s', 'RPM', 'Wind_Gust_mps', 'Turbulence_Intensity', 'Blade_Pitch_Deg']
).copy()

palette = px.colors.qualitative.Plotly
colors = {
    tid: palette[i % len(palette)]
    for i, tid in enumerate(sorted(relationship_df['Turbine_ID'].unique()))
}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Power curve: wind speed vs power',
        'Turbulence vs vibration',
        'Wind gusts vs RPM',
        'Blade pitch vs power',
    ),
)

fit_rows = []
for tid, group in relationship_df.groupby('Turbine_ID'):
    color = colors[tid]

    # Power curve
    fig.add_trace(go.Scatter(x=group['Wind_Speed_mps'], y=group['Power_kW'], mode='markers',
        marker=dict(color=color, size=4, opacity=0.5), name=tid, legendgroup=tid,
        hovertemplate='Wind=%{x:.1f}<br>Power=%{y:.0f}<extra>' + tid + '</extra>'), row=1, col=1)
    pc = np.polyfit(group['Wind_Speed_mps'], group['Power_kW'], 2)
    xg = np.linspace(group['Wind_Speed_mps'].min(), group['Wind_Speed_mps'].max(), 100)
    fig.add_trace(go.Scatter(x=xg, y=np.polyval(pc, xg), mode='lines',
        line=dict(color=color, width=2, dash='dash'), legendgroup=tid, showlegend=False, hoverinfo='skip'), row=1, col=1)

    # Turbulence vs vibration
    fig.add_trace(go.Scatter(x=group['Turbulence_Intensity'], y=group['Vibration_mm_s'], mode='markers',
        marker=dict(color=color, size=4, opacity=0.5), legendgroup=tid, showlegend=False,
        hovertemplate='Turb=%{x:.3f}<br>Vib=%{y:.2f}<extra>' + tid + '</extra>'), row=1, col=2)
    vc = np.polyfit(group['Turbulence_Intensity'], group['Vibration_mm_s'], 1)
    vg = np.linspace(group['Turbulence_Intensity'].min(), group['Turbulence_Intensity'].max(), 100)
    fig.add_trace(go.Scatter(x=vg, y=np.polyval(vc, vg), mode='lines',
        line=dict(color=color, width=2, dash='dash'), legendgroup=tid, showlegend=False, hoverinfo='skip'), row=1, col=2)

    # Gusts vs RPM
    fig.add_trace(go.Scatter(x=group['Wind_Gust_mps'], y=group['RPM'], mode='markers',
        marker=dict(color=color, size=4, opacity=0.5), legendgroup=tid, showlegend=False,
        hovertemplate='Gust=%{x:.2f}<br>RPM=%{y:.1f}<extra>' + tid + '</extra>'), row=2, col=1)
    rc = np.polyfit(group['Wind_Gust_mps'], group['RPM'], 1)
    rg = np.linspace(group['Wind_Gust_mps'].min(), group['Wind_Gust_mps'].max(), 100)
    fig.add_trace(go.Scatter(x=rg, y=np.polyval(rc, rg), mode='lines',
        line=dict(color=color, width=2, dash='dash'), legendgroup=tid, showlegend=False, hoverinfo='skip'), row=2, col=1)

    # Blade pitch vs power
    fig.add_trace(go.Scatter(x=group['Blade_Pitch_Deg'], y=group['Power_kW'], mode='markers',
        marker=dict(color=color, size=4, opacity=0.5), legendgroup=tid, showlegend=False,
        hovertemplate='Pitch=%{x:.1f}<br>Power=%{y:.0f}<extra>' + tid + '</extra>'), row=2, col=2)
    bpc = np.polyfit(group['Blade_Pitch_Deg'], group['Power_kW'], 1)
    bpg = np.linspace(group['Blade_Pitch_Deg'].min(), group['Blade_Pitch_Deg'].max(), 100)
    fig.add_trace(go.Scatter(x=bpg, y=np.polyval(bpc, bpg), mode='lines',
        line=dict(color=color, width=2, dash='dash'), legendgroup=tid, showlegend=False, hoverinfo='skip'), row=2, col=2)

    fit_rows.append({'Turbine_ID': tid,
        'Power_at_10mps': round(float(np.polyval(pc, 10.0)), 1),
        'Vibration_slope': round(float(vc[0]), 2),
        'RPM_gust_slope': round(float(rc[0]), 2),
        'Pitch_power_slope': round(float(bpc[0]), 2)})

fig.update_layout(template='plotly_white', height=900, title='Relationship checks: four physics-based views')
fig.update_xaxes(title_text='Wind speed (m/s)', row=1, col=1)
fig.update_yaxes(title_text='Power (kW)', row=1, col=1)
fig.update_xaxes(title_text='Turbulence intensity', row=1, col=2)
fig.update_yaxes(title_text='Vibration (mm/s)', row=1, col=2)
fig.update_xaxes(title_text='Wind gust (m/s)', row=2, col=1)
fig.update_yaxes(title_text='RPM', row=2, col=1)
fig.update_xaxes(title_text='Blade pitch (deg)', row=2, col=2)
fig.update_yaxes(title_text='Power (kW)', row=2, col=2)
fig

In [ ]:
print(pd.DataFrame(fit_rows).to_string(index=False))

### [TEACHER] Reading the four-panel chart and the fit table

**Power curve (top-left):**
> "A healthy turbine's power curve rises steeply from cut-in wind speed (~3 m/s), then flattens when it hits rated output. Look at the fitted curve for each turbine. If one turbine's curve is much lower or flatter than the others at the same wind speed, it is losing efficiency — power is going somewhere other than the grid."

> "Check the `Power_at_10mps` column in the table. WT-01 should be highest. WT-02 should be noticeably lower — the seizure means it spent most of day 45 onward producing nothing. WT-03 should be close to WT-01 at this point, which is why it is the subtle one."

**Turbulence vs vibration (top-right):**
> "Higher turbulence loading means more structural vibration — that relationship should be positive for all turbines. If one turbine's slope is much steeper, it is responding more strongly to the same wind input. That is consistent with a bearing or drivetrain that is not damping as well as it should."

**Gusts vs RPM (bottom-left):**
> "Gusts add a short transient to the rotor loading. A healthy machine tracks gusts fairly tightly — the slope should be similar across the three turbines. A degraded machine might overshoot, undershoot, or show more scatter around the trend."

**Blade pitch vs power (bottom-right):**
> "This one is often the most diagnostic. In normal operation, pitch and power are loosely linked — the controller adjusts pitch to optimise power delivery. A machine that is offline or in a failure state will show pitch locked at its feathered angle (90°) while power is zero. Look for WT-02's cluster of points: all at the same pitch, zero power."

**Prompt:** *Which turbine shows the weakest pitch-vs-power relationship? What does that mean operationally?*

---
## Section 3 — Vibration Normalised by RPM

### [TEACHER] Why we normalise — what to say

> "Here is the problem. Vibration goes up with RPM. A turbine spinning faster produces more vibration even if it is perfectly healthy — it is a physics effect. So if we just look at raw vibration over time, we cannot tell whether a rise is because the wind picked up or because the machine is getting rougher."

> "The fix is simple: divide vibration by RPM. Now we are asking: for each rotation, how much vibration? A smooth, healthy bearing should produce a stable ratio. A degraded bearing produces more vibration per rotation as it gets rougher. The ratio is your early warning signal."

> "There is one catch: we cannot divide by zero. If the turbine is below cut-in wind speed, RPM might be very low (near 1–2 RPM). Dividing by those values inflates the ratio and creates noise. We handle that with `replace(0, np.nan)` — any exact zero becomes a missing value. Pandas will skip NaN in the rolling mean."

**Give participants a minute to try computing the ratio before showing the code.**

In [ ]:
# Compute vibration-per-RPM for WT-03 specifically
# Replace exact zeros before dividing (near-zero RPM below cut-in)
wt03_df = analysis_df[analysis_df['Turbine_ID'] == 'WT-03'].sort_values('Timestamp').copy()

wt03_df['Vibration_per_RPM'] = wt03_df['Vibration_mm_s'] / wt03_df['RPM'].replace(0, np.nan)

print('Vibration/RPM stats for WT-03:')
print(wt03_df['Vibration_per_RPM'].describe().round(4))

In [ ]:
# Rolling mean with a 24-step window (24 × 10 min = 4 hours)
# min_periods=6 means we still get a rolling value after the first hour of data
wt03_df['Vibration_per_RPM_Rolling'] = (
    wt03_df['Vibration_per_RPM'].rolling(window=24, min_periods=6).mean()
)

fig = px.line(
    wt03_df,
    x='Timestamp',
    y='Vibration_per_RPM',
    title='WT-03: Vibration Normalised by RPM (with 4-hour rolling mean)',
    hover_data=['Bearing_Temperature_C', 'Vibration_mm_s', 'RPM', 'Wind_Speed_mps', 'Wind_Gust_mps'],
)
fig.add_scatter(
    x=wt03_df['Timestamp'],
    y=wt03_df['Vibration_per_RPM_Rolling'],
    mode='lines',
    name='4-hour rolling mean',
    line=dict(width=3),
)
fig.update_layout(
    template='plotly_white',
    xaxis_title='Timestamp',
    yaxis_title='Vibration per RPM',
)
fig

### [TEACHER] What to say when the rolling mean appears

> "The raw ratio (the thin line) is noisy because each 10-minute reading is affected by whatever the wind was doing at that moment. The rolling mean (thick line) smooths that out over a 4-hour window — long enough to damp weather noise, short enough to still show multi-day trends."

> "Now look at the shape of the rolling mean. In the first 60 days, is it flat? After day 60, does it start to rise? That rise — even a small one — is the early drift signal. It is not a spike, it is a slope change."

> "This is the diagnostic that would justify a maintenance referral in a real condition monitoring system. Not the peak vibration. The trend in the normalised ratio."

**Prompt:** *Compare this to the same ratio for WT-01 — it should be flat across the whole 90 days. If WT-01 also rises, we have to consider whether weather changed for all turbines. If only WT-03 rises, we have asset-specific deterioration.*

**Tip:** You can add a WT-01 trace here to make the comparison immediate. Keep it as a prompt rather than giving them the code directly — let them add it.

In [ ]:
# Add WT-01 to the same plot as a healthy baseline comparison
wt01_df = analysis_df[analysis_df['Turbine_ID'] == 'WT-01'].sort_values('Timestamp').copy()
wt01_df['Vibration_per_RPM'] = wt01_df['Vibration_mm_s'] / wt01_df['RPM'].replace(0, np.nan)
wt01_df['Vibration_per_RPM_Rolling'] = wt01_df['Vibration_per_RPM'].rolling(window=24, min_periods=6).mean()

fig.add_scatter(
    x=wt01_df['Timestamp'],
    y=wt01_df['Vibration_per_RPM_Rolling'],
    mode='lines',
    name='WT-01 rolling mean (healthy baseline)',
    line=dict(dash='dot', width=2),
)
fig.update_layout(title='Vibration/RPM: WT-03 vs WT-01 healthy baseline')
fig

### [TEACHER] The comparison with WT-01

> "If WT-01's ratio stays flat while WT-03's rises, that rules out wind as the explanation. Same site, same weather, different behaviour. The difference has to come from the machine."

> "This is the argument you take to a maintenance manager: not 'vibration went up', but 'vibration per rotation went up on one turbine while the identical turbine next to it stayed stable, and the only thing that changed was time'. That is a much harder case to dismiss."

---
## Section 4 — The Operational Decision and Dashboard Ethics

### [TEACHER] Framing the decision before writing any code

> "You have one maintenance slot this quarter. Three turbines. WT-01 is healthy. WT-02 is already offline — it has failed. WT-03 is drifting. The budget question is: do you spend the money on WT-02, which is already down and losing revenue, or on WT-03, which is still running but heading in the wrong direction?"

> "To make that recommendation from data you need two things: a threshold — a value at which we would say 'this is unacceptable' — and a projected date when WT-03 will reach that threshold."

> "For the threshold we use the healthy baseline: WT-01's mean vibration/RPM ratio plus 2 standard deviations. For the projection we fit a linear trend to WT-03's ratio and see where it crosses the line."

**Before building the dashboard, ask participants:** *What else might influence this decision that the data alone cannot answer?* Good answers: cost to repair vs cost of an outage, safety risk if WT-03 seizes, whether the maintenance team has the right skills, contract penalties for downtime. Make sure someone mentions safety and environmental impact before you move on.

In [ ]:
# Step 1: Establish the warning threshold from the healthy turbine
# Use WT-01 as the reference — mean + 2 standard deviations
analysis_df['Vibration_per_RPM'] = (
    analysis_df['Vibration_mm_s'] / analysis_df['RPM'].replace(0, np.nan)
)

healthy_baseline = analysis_df.loc[
    analysis_df['Turbine_ID'] == 'WT-01', 'Vibration_per_RPM'
].dropna()

warning_threshold = healthy_baseline.mean() + 2 * healthy_baseline.std()
print(f'WT-01 mean vibration/RPM: {healthy_baseline.mean():.4f}')
print(f'WT-01 std vibration/RPM:  {healthy_baseline.std():.4f}')
print(f'Warning threshold (mean + 2 SD): {warning_threshold:.4f}')

### [TEACHER] What to say about the threshold

> "Mean plus 2 standard deviations covers roughly 95 % of normal behaviour. Any reading above that threshold is unusual enough to flag. It is not a physics-derived limit — it is a statistical one, calibrated to what WT-01 actually does."

> "If a maintenance team has a manufacturer's vibration limit (they often do — it might be in the O&M manual), you should use that instead. This is our best available proxy when we do not have that document."

> "Write this number down. It is the horizontal line that will go on the final chart."

In [ ]:
# Step 2: Fit a linear trend to WT-03's vibration/RPM ratio and project crossing
wt03_ratio = analysis_df.loc[
    analysis_df['Turbine_ID'] == 'WT-03',
    ['Timestamp', 'Vibration_per_RPM', 'Wind_Speed_mps', 'Wind_Gust_mps', 'Turbulence_Intensity', 'Bearing_Temperature_C'],
].dropna().copy()

wt03_ratio['Day_Index'] = (
    (wt03_ratio['Timestamp'] - wt03_ratio['Timestamp'].min()).dt.total_seconds() / 86400
)

ratio_coeffs = np.polyfit(wt03_ratio['Day_Index'], wt03_ratio['Vibration_per_RPM'], deg=1)
wt03_ratio['Trend_Fit'] = np.polyval(ratio_coeffs, wt03_ratio['Day_Index'])

slope_per_day = ratio_coeffs[0]
print(f'Trend slope: {slope_per_day:.6f} vibration/RPM per day')

if slope_per_day > 0:
    projected_day = (warning_threshold - ratio_coeffs[1]) / slope_per_day
    projected_date = wt03_ratio['Timestamp'].min() + pd.to_timedelta(projected_day, unit='D')
    print(f'Projected threshold crossing: {projected_date.strftime("%Y-%m-%d")}')
else:
    projected_date = None
    print('No positive trend — projection not applicable')

### [TEACHER] What to say about the trend projection

> "The slope is small — a fraction of a unit per day. But compounded over 30 days it adds up. The projected crossing date tells us: if the current rate continues, when will WT-03's behaviour be indistinguishable from WT-02 before it failed?"

> "A linear fit is a simplification. Real degradation often accelerates as it progresses — think of metal fatigue. A linear projection is an optimistic lower bound. If the slope accelerates, the crossing happens sooner."

> "This is the number that goes in the maintenance referral: 'Based on the current trend, WT-03 is projected to reach the vibration warning threshold on [date]. We recommend scheduling an inspection before that date.'"

**Tip:** If participants ask why linear and not exponential — both are defensible. Linear is the minimum defensible assumption with the available data. You would fit exponential if you had more evidence of acceleration. Add that to the referral as a caveat: 'Projection assumes constant rate of deterioration.'

In [ ]:
# Step 3: Build the two-panel evidence dashboard
power_column = 'Power_kW'
power_df = analysis_df.dropna(subset=['Wind_Speed_mps', power_column, 'Blade_Pitch_Deg']).copy()

palette = px.colors.qualitative.Plotly
colors = {
    tid: palette[i % len(palette)]
    for i, tid in enumerate(sorted(power_df['Turbine_ID'].unique()))
}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'WT-03 vibration per RPM — trend and warning line',
        'Power output vs wind speed by turbine',
    ),
)

# Left panel: WT-03 trend with warning line
fig.add_trace(
    go.Scatter(
        x=wt03_ratio['Timestamp'], y=wt03_ratio['Vibration_per_RPM'],
        mode='markers', name='WT-03 observed',
        marker=dict(color=colors['WT-03'], size=4, opacity=0.4),
        hovertemplate='%{x}<br>Vib/RPM=%{y:.4f}<extra>WT-03</extra>',
    ), row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=wt03_ratio['Timestamp'], y=wt03_ratio['Trend_Fit'],
        mode='lines', name='WT-03 trend',
        line=dict(color=colors['WT-03'], width=3),
        hoverinfo='skip',
    ), row=1, col=1
)
fig.add_hline(y=warning_threshold, line_dash='dot', line_color='firebrick', row=1, col=1)
fig.add_annotation(
    x=wt03_ratio['Timestamp'].iloc[len(wt03_ratio) // 4],
    y=warning_threshold,
    text=f'Warning threshold: {warning_threshold:.3f}',
    showarrow=False, yshift=12, row=1, col=1,
)
if projected_date is not None:
    fig.add_vline(x=projected_date, line_dash='dash', line_color='firebrick', row=1, col=1)

# Right panel: power curves for all three turbines
for tid, group in power_df.groupby('Turbine_ID'):
    color = colors[tid]
    fig.add_trace(
        go.Scatter(
            x=group['Wind_Speed_mps'], y=group[power_column],
            mode='markers', name=f'{tid} power',
            marker=dict(color=color, size=4, opacity=0.35),
            hovertemplate='Wind=%{x:.1f}<br>Power=%{y:.0f}<extra>' + tid + '</extra>',
            legendgroup=tid,
        ), row=1, col=2
    )
    pc = np.polyfit(group['Wind_Speed_mps'], group[power_column], 2)
    xg = np.linspace(group['Wind_Speed_mps'].min(), group['Wind_Speed_mps'].max(), 120)
    fig.add_trace(
        go.Scatter(
            x=xg, y=np.polyval(pc, xg),
            mode='lines', name=f'{tid} fit',
            line=dict(color=color, width=2, dash='dash'),
            legendgroup=tid, showlegend=False, hoverinfo='skip',
        ), row=1, col=2
    )

fig.update_layout(
    template='plotly_white', height=550,
    title='Evidence dashboard: trend fit and power output comparison',
)
fig.update_xaxes(title_text='Timestamp', row=1, col=1)
fig.update_yaxes(title_text='Vibration per RPM', row=1, col=1)
fig.update_xaxes(title_text='Wind speed (m/s)', row=1, col=2)
fig.update_yaxes(title_text='Power (kW)', row=1, col=2)
fig

In [ ]:
# Concise summary table for the maintenance decision
summary_rows = []
for tid, group in analysis_df.groupby('Turbine_ID'):
    vpr = group['Vibration_per_RPM'].dropna()
    summary_rows.append({
        'Turbine': tid,
        'Mean power (kW)': round(group['Power_kW'].mean(), 1),
        'Max vibration/RPM': round(vpr.max(), 4),
        'Latest Maintenance_Flag': group['Maintenance_Flag'].iloc[-1],
        'Status': group['Operational_State'].iloc[-1],
    })

summary = pd.DataFrame(summary_rows)
print(summary.to_string(index=False))
print()
if projected_date is not None:
    print(f'WT-03 projected threshold crossing: {projected_date.strftime("%Y-%m-%d")}')
print(f'Recommendation: prioritise WT-03 — the only turbine where proactive intervention is still possible')

### [TEACHER] Making the recommendation — what to say

> "The table tells us three things clearly: WT-01 is fine, WT-02 has failed, WT-03 is still running but drifting. The only decision left is whether the maintenance budget goes to WT-02 (repair a known failure) or WT-03 (prevent a predicted failure)."

> "From a purely operational standpoint, WT-02 is a sunk cost — it is already offline. The revenue it was generating is already lost. Spending money to restore it gets you back to baseline. Spending money to prevent WT-03 failing avoids a future outage, a future emergency repair (which typically costs more than planned maintenance), and the risk of a worse failure if the seizure happens during a storm."

> "The data recommendation is WT-03. The projected threshold date is the deadline for the intervention."

**Important caveat to say out loud:** *'This recommendation assumes the data is correct. We used a statistical threshold, not a physics-based one. If the vibration sensor on WT-03 has any bias or drift, the trend could be artefact rather than signal. Before scheduling the maintenance, you should validate the sensor reading against another instrument or do a physical inspection. The dashboard justifies an inspection — it does not replace one.'*

---
## [TEACHER] Ethics and Governance Discussion (5 minutes)

This is the section to run as a whole-group discussion. Do not write more code — use the summary table on screen and ask these questions in order.

**Question 1 — Safety and liability:**
> "If WT-03 seizes while we are still deciding whether to intervene, and a technician is nearby when it fails, who is responsible? The data showed the trend. The recommendation was made. Who should have acted on it?"

**Question 2 — Environmental impact:**
> "A seized bearing can cause vibration that damages the tower foundation over time. A catastrophic seizure at speed can release lubricating oil. How does this change the cost calculation beyond lost revenue?"

**Question 3 — Dashboard design:**
> "This dashboard shows the trend but not the uncertainty. If the trend slope has high uncertainty — the R² is low — the projection could be wildly wrong. How should we communicate that on a chart that a non-technical manager will read?"

**Question 4 — Data quality governance:**
> "We removed 3 % of RPM and Power rows. We applied an IQR filter. We kept the failure records in a separate stream. None of that is visible in the final dashboard. If someone downstream uses this dashboard to make a decision, should they know about those choices?"

**Expected direction:** Yes. Every analysis choice that could affect the conclusion — what you dropped, what you assumed, what threshold you set — belongs in a methodology note alongside the dashboard. Visualisation design is an ethical act: what you choose to show and hide determines what decisions get made.

---
## Wrap-Up — What Participants Should Now Be Able to Do

By the end of this session, a participant who completed the exercise should be able to:

1. **Load and inspect** a messy real-world telemetry CSV and articulate what the missing values mean before touching them
2. **Split into two data streams** — forensic (dropout rows) and statistical (cleaned rows) — and explain why ffill is the wrong default
3. **Build four relationship scatter plots** and read them for signs that a turbine is drifting from expected physics
4. **Normalise vibration by RPM** to separate speed effects from roughness, and add a rolling mean to reveal slow trends
5. **Fit a linear trend**, set a threshold from a healthy baseline, and project a crossing date
6. **Articulate** the maintenance recommendation in terms the data supports — and name at least two things the data alone cannot settle

### For the teacher — common places participants get stuck

- **ffill vs dropna**: Many people default to ffill because it feels tidier. Ask them what ffill does to a 24-hour sensor dropout — it fills the outage window with the last good reading, hiding the failure completely.
- **The scatter is overplotted**: Reduce `size_max`, add `opacity=0.5`, or filter to a turbine before plotting. Having too many points at full opacity makes the outliers invisible.
- **The rolling mean looks flat**: The window might be too wide (averaging out the signal) or the column might include NaN rows that are breaking the rolling window. Check `min_periods` and make sure the data is sorted by Timestamp before rolling.
- **The trend slope is near-zero**: This can happen if the IQR filter was too aggressive and removed the high-vibration WT-03 rows in month 3. Check `analysis_df['Degradation_Factor'].describe()` to see if the drift period is represented.